In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import pandas as pd # for opening csv files

from pathlib import Path      # used to play with pathnames to save 

# import xarray as xr

# import zipfile as zp          # used for unzipping ppi files
# from pathlib import Path      # used to play with pathnames to save 
# from datetime import datetime # used to manipulate time :)


# import wradlib as wr          # used for having fun with radar data

# from PIL import Image         # used for creating gif loops
# import os                     # used for retrieving file names

In [ ]:
# import the csv files into a large python data array

# create an empty array for storing the mean wind in the lowest layer
LowMeanWind = np.full([367], np.nan) #one more day than the leap year 2024 since the first entry (day 0) is Nan

MonthLengths = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31] # number of days in each month of the year

Station = 'Townsville'

if (Station == 'Townsville'):
    StationID = 95282

MaxLowHeight = 5000 # [m] top height of where to consider the mean wind of the lowest layer


year = 2024

for monthi in range(1,13):
    for dayi in range(1,MonthLengths[monthi-1] + 1):

        doyi = int(np.sum(MonthLengths[0:monthi-1]) + dayi)

        hour = 0
        
        YYYY = str(year).zfill(4)
        MM = str(monthi).zfill(2)
        DD = str(dayi).zfill(2)
        
        hh = str(hour).zfill(2)
        
        SoundingsFolder = '/home/563/sg3241/TownsvilleSoundings/'
        SoundingsFile = YYYY + MM + DD + hh + '-' + str(StationID) + '.csv'
        
        SoundingPath = SoundingsFolder + SoundingsFile



        if not Path(SoundingPath).is_file():
            print('No file found for ' + SoundingPath)
        else:
            WorkingSounding = pd.read_csv(SoundingPath, usecols=['geopotential height_m', 'wind direction_degree', 'wind speed_m/s'])

            rad = np.deg2rad(WorkingSounding['wind direction_degree'])
            WorkingSounding['u'] = -WorkingSounding['wind speed_m/s'] * np.sin(rad)   # eastward
            WorkingSounding['v'] = -WorkingSounding['wind speed_m/s'] * np.cos(rad)   # northward

        
            # calculate the mean wind (CAUTION mean of the data points, not weighted by depth) in the lowest ZZZZ metres
            LowMeanWind[doyi] = WorkingSounding[WorkingSounding['geopotential height_m'] < MaxLowHeight]['u'].mean()

In [ ]:
x = np.arange(1,368,1)

fig, ax = plt.subplots(figsize=(16,4))
plt.plot(x,LowMeanWind,color=[0.5,0,0])

plt.xlim([0,366])
plt.ylim([-15,15])

plt.xlabel('Day of Year [2024]')
plt.ylabel('Mean Wind [m/s]')

plt.title('Townsville Soundings: Mean Zonal Wind Speed in the lowest ' + str(MaxLowHeight) + ' Metres')

# Vertical lines: thick every 30 days, thin every 5 days
for day in range(0, 367, 5):
    linewidth = 1 if day % 30 == 0 else 0.1
    ax.axvline(day, color='k', linewidth=linewidth, alpha=0.7)

# Horizontal lines: thick at 0, thin every 2 m/s
for wind in range(-16, 16, 2):
    linewidth = 1 if wind == 0 else 0.3
    ax.axhline(wind, color='k', linewidth=linewidth, alpha=0.7)